# 컨텍스트, 하네스, 루프: 구현하기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 들어가며

이 레슨은 네 층위를 코드로 확인합니다. chat template을 해부해 실제 전달 문자열을 보고 컨텍스트 예산을 잽니다. 이어서 중간 소실을 재현하고 센서를 붙여 품질 변화를 잰 뒤 검증 루프가 수렴하는 모습을 봅니다.

#### 실행 환경

이 레슨은 **GPU 런타임**이 필요합니다. 실습 환경 GPU(T4, VRAM 16GB)에서 처음부터 끝까지 돌아갑니다. 런타임 유형이 GPU인지 확인하고 시작하십시오.

모델은 **하나만 올립니다.** Qwen2.5-7B-Instruct를 4비트로 올려 끝까지 씁니다. Step 1에서 받는 Llama 토크나이저는 GPU를 쓰지 않습니다.

**API 키는 필요 없습니다.** 백엔드를 `"endpoint"`로 바꿀 때만 엔드포인트 주소가 필요하고 그것은 🖥 로컬 작업입니다.

PC나 사내 서버에서 해야 하는 일에는 🖥 **로컬 작업** 표시를 붙였습니다. 실습 환경에서는 읽고 넘어가도 됩니다.

이 레슨은 **같은 질문을 수십 번 다시 생성하는 것이 실습 내용**이라 오래 걸리는 셀이 여럿 있습니다.

시간이 모자라면 Step 3의 `N_FILLER`와 `PROBE_TRIALS`를 줄이십시오. 줄이면 무엇이 흐려지는지는 해당 셀 주석에 적어 두었습니다.

실습 셀(코드가 비어 있고 바로 아래 '정답 보기'가 붙은 셀)의 저장된 출력은 정답 코드를 넣고 실행한 결과입니다. 코드를 채우기 전에 그대로 실행하면 다른 결과가 나옵니다.

## Step 0 : 환경 준비

In [ ]:
# 버전을 고정합니다. 상한이 없으면 어느 날 메이저 버전이 올라와 어제 돌던 코드가 멈춥니다
!pip install -q "transformers>=5.17,<6" "accelerate>=1.15,<2" "bitsandbytes>=0.50,<0.51"
!pip install -q "openai>=3.16,<4"

# 설치 후 pip이 의존성 충돌을 보고하면 [런타임] - [세션 다시 시작]을 한 번 누르고
# 이 셀 아래부터 다시 실행하세요

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.1 MB/s eta 0:00:00


In [ ]:
import torch, json, re, time, os
from importlib.metadata import version, PackageNotFoundError

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("GPU가 없습니다. [런타임] - [런타임 유형 변경]에서 GPU를 선택하세요")


def pkgver(dist):
    """배포판 이름으로 설치된 버전을 읽습니다"""
    try:
        return version(dist)
    except PackageNotFoundError:
        return "미설치"


# 설치된 버전을 찍어둡니다. 나중에 결과가 이상하면 이 숫자부터 확인합니다
print("\n[설치된 버전]")
for d in ["transformers", "accelerate", "bitsandbytes", "openai"]:
    print(f"  {d:16s} {pkgver(d)}")

GPU : Tesla T4
VRAM : 15.6 GB

[설치된 버전]
  transformers     5.17.0
  accelerate       1.15.0
  bitsandbytes     0.50.2
  openai           3.24.0

앞 노드에서 만든 백엔드 추상화를 그대로 씁니다. 로컬 GPU와 원격 엔드포인트를 상수 하나로 전환합니다.

🖥 **로컬 작업** : `BACKEND = "endpoint"` 쪽은 사내 GPU 서버에 추론 서버를 띄워 둔 경우입니다. 실습 환경에서는 `"local"`로 두고 읽기만 하십시오.

In [ ]:
BACKEND = "local"                       # "local" 또는 "endpoint"
ENDPOINT_URL = "http://localhost:8000/v1"
ENDPOINT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
ENDPOINT_KEY = "dummy-key"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
# 토크나이저는 두 백엔드가 모두 씁니다. 분기 밖에서 임포트합니다.
# 안에 두면 "endpoint"로 바꾸는 순간 AutoTokenizer가 없다는 오류가 납니다
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if BACKEND == "local":
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    _model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                                 dtype=torch.float16,   # 양자화되지 않는 계층의 정밀도. T4는 bf16 미지원이라 명시합니다
                                                 device_map="auto")
    _model.eval()
    print(f"로컬 모델 로드 완료. VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    from openai import OpenAI
    _client = OpenAI(base_url=ENDPOINT_URL, api_key=ENDPOINT_KEY)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # 토큰 계산용으로만 사용
    print("엔드포인트 준비 완료")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

로컬 모델 로드 완료. VRAM 5.56 GB

In [ ]:
def chat(messages, max_new_tokens=512, temperature=0.2):
    if BACKEND == "endpoint":
        r = _client.chat.completions.create(model=ENDPOINT_MODEL, messages=messages,
                                           temperature=temperature, max_tokens=max_new_tokens)
        return r.choices[0].message.content or ""
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, return_tensors="pt").to(_model.device)
    # temperature를 do_sample=False와 함께 넘기면 호출마다 경고가 뜹니다.
    # 샘플링을 켤 때만 넘깁니다
    gen = {"max_new_tokens": max_new_tokens, "pad_token_id": tokenizer.eos_token_id}
    gen.update({"do_sample": True, "temperature": temperature} if temperature > 0
               else {"do_sample": False})
    with torch.no_grad():
        out = _model.generate(**ids, **gen)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

## Step 1 : chat template 해부

이론 레슨에서 컨텍스트를 설계한다는 것은 모델에게 실제로 전달되는 문자열을 통제한다는 뜻이라고 했습니다. 그런데 우리는 지금까지 `messages` 리스트만 넘겼습니다. 그 리스트가 어떤 문자열로 바뀌는지 봅니다.

In [ ]:
messages = [
    {"role": "system", "content": "당신은 사내 규정 안내 담당자입니다."},
    {"role": "user", "content": "연차 이월이 가능한가요?"},
    {"role": "assistant", "content": "원칙적으로는 불가능합니다."},
    {"role": "user", "content": "예외는 없나요?"},
]

rendered = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print(rendered)

<|im_start|>system
당신은 사내 규정 안내 담당자입니다.<|im_end|>
<|im_start|>user
연차 이월이 가능한가요?<|im_end|>
<|im_start|>assistant
원칙적으로는 불가능합니다.<|im_end|>
<|im_start|>user
예외는 없나요?<|im_end|>
<|im_start|>assistant

`<|im_start|>`, `<|im_end|>` 같은 특수 토큰이 역할을 구분합니다. 마지막에 붙은 `<|im_start|>assistant`는 `add_generation_prompt=True`의 효과로 모델에게 "이제 assistant 차례다"라고 알려주는 신호입니다.

이 신호를 빼 봅니다.

In [ ]:
without_gen = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
print(repr(without_gen[-120:]))

'_start|>user\n연차 이월이 가능한가요?<|im_end|>\n<|im_start|>assistant\n원칙적으로는 불가능합니다.<|im_end|>\n<|im_start|>user\n예외는 없나요?<|im_end|>\n'

In [ ]:
# 생성 프롬프트 없이 생성하면 어떻게 되는지 확인합니다
ids = tokenizer(without_gen, return_tensors="pt").to(_model.device)
with torch.no_grad():
    out = _model.generate(**ids, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

 companny policy allows for a maximum of 3 days of accumulated unused annual leave to be carried over to the next year. If an employee has more than 3 days, those excess days must be taken in the current year or they will be forfeited.

모델이 assistant 답변 대신 엉뚱한 것을 이어 쓰거나 사용자 발화를 계속 만들어 내기도 하고, 위 출력처럼 다른 언어로 답을 이어 쓰기도 합니다. **에러는 나지 않고** 품질만 떨어집니다.

독점형 API는 서버가 이 층위를 처리해 주지만 프라이빗 환경에서 오픈소스 모델을 직접 다룰 때는 우리 몫입니다.

모델마다 템플릿이 달라서 Qwen과 Llama를 나란히 놓고 비교합니다. `meta-llama` 계정의 저장소는 허깅페이스에서 약관에 동의하고 토큰으로 로그인해야 내려받을 수 있는 **게이트(gated) 저장소**입니다. 그래서 템플릿 비교 셀은 게이트가 없는 미러 저장소를 먼저 시도하고 실패하면 원본을 시도하고 둘 다 실패해도 멈추지 않습니다.

In [ ]:
# 모델마다 템플릿이 다릅니다. 두 모델을 비교해봅시다
# meta-llama는 게이트 저장소라 로그인이 필요합니다. 게이트 없는 미러를 먼저 시도합니다
from transformers import AutoTokenizer

tok_qwen = tokenizer
tok_llama = None
for repo in ["unsloth/Llama-3.2-1B-Instruct",        # 게이트 없음. 템플릿은 원본과 같습니다
             "meta-llama/Llama-3.2-1B-Instruct"]:    # 원본. 로그인이 되어 있으면 이쪽이 받아집니다
    try:
        tok_llama = AutoTokenizer.from_pretrained(repo)
        print(f"Llama 토크나이저 로드 : {repo}")
        break
    except Exception as e:
        print(f"건너뜀 : {repo} ({type(e).__name__})")

simple = [{"role": "system", "content": "You are helpful."},
          {"role": "user", "content": "Hi"}]

print("\n=== Qwen2.5 ===")
print(tok_qwen.apply_chat_template(simple, add_generation_prompt=True, tokenize=False))
print("\n=== Llama 3.2 ===")
if tok_llama is not None:
    print(tok_llama.apply_chat_template(simple, add_generation_prompt=True, tokenize=False))
else:
    print("(토크나이저를 받지 못했습니다. 아래 설명의 예시 출력을 보세요)")

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB

tokenizer.json: downloading bytes:           |  0.00B

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Llama 토크나이저 로드 : unsloth/Llama-3.2-1B-Instruct

=== Qwen2.5 ===
<|im_start|>system
You are helpful.<|im_end|>
<|im_start|>user
Hi<|im_end|>
<|im_start|>assistant


=== Llama 3.2 ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Oct 2026

You are helpful.<|eot_id|><|start_header_id|>user<|end_header_id|>

Hi<|eot_id|><|start_header_id|>assistant<|end_header_id|>

구분자가 완전히 다릅니다. 두 모델이 실제로 받는 문자열은 이런 모양입니다.

```
Qwen2.5   <|im_start|>system\nYou are helpful.<|im_end|>\n<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n

Llama 3.2 <|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are helpful.<|eot_id|>
          <|start_header_id|>user<|end_header_id|>\n\nHi<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n
```

한 모델의 템플릿에 맞춰 다듬어 둔 프롬프트를 다른 모델로 그대로 옮기면 성능이 떨어지기도 합니다.

Llama 템플릿은 시스템 메시지 앞에 지식 기준일과 오늘 날짜(`Cutting Knowledge Date`, `Today Date`)를 자동으로 끼워 넣습니다. 위 출력에서 확인하십시오. 날짜가 매일 바뀌므로 이론 레슨 3.7절의 프리픽스 캐시 관점에서는 프롬프트 앞부분이 매일 달라지는 셈입니다.

원본 저장소를 쓰고 싶다면 허깅페이스에서 해당 모델 페이지의 약관에 동의한 뒤 아래를 실행하고 셀을 다시 돌리면 됩니다.

```python
from huggingface_hub import login
login()        # 토큰 입력창이 뜹니다
```

## Step 2 : 컨텍스트 예산 측정

컨텍스트 윈도우를 예산으로 보고 집행 내역서를 만듭니다. 한 번의 호출에서 무엇이 몇 토큰을 쓰는지 계산합니다.

In [ ]:
%%writefile policy.md
## 제3조 (근무 시간)
기본 근무 시간은 주 40시간으로 하며, 1일 근무 시간은 휴게 시간을 제외하고 8시간으로 한다. 시업 시각은 오전 10시, 종업 시각은 오후 7시로 하되, 유연근무제 적용자는 제4조에 따른다.

## 제4조 (유연근무제)
임직원은 오전 8시부터 오전 11시 사이에 시업 시각을 자율적으로 선택할 수 있다. 코어 타임은 오전 11시부터 오후 4시까지로 한다.

## 제5조 (재택근무)
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.

## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

## 제8조 (경조 휴가)
본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 본인 및 배우자의 부모 사망 5일, 조부모 및 형제자매 사망 3일을 부여한다.

## 제10조 (사내 데이터의 외부 반출)
고객 개인정보, 미공개 재무 정보, 기술 설계 문서를 포함한 사내 데이터는 정보보호팀장의 사전 승인 없이 외부로 반출할 수 없다.

Writing policy.md

In [ ]:
with open("policy.md", encoding="utf-8") as f:
    doc = f.read()
chunks = [c.strip() for c in doc.split("## ") if c.strip()]
chunks = ["## " + c for c in chunks]

SYSTEM = """당신은 사내 규정 안내 담당자입니다.
제공된 규정에 근거해서만 답변하고, 근거 조항을 반드시 표기하세요.
규정에 없는 내용이면 확인되지 않는다고 답하세요.
답변은 반드시 한국어로만 작성하세요."""

TOOLS_DEF = json.dumps([
    {"name": "search_policy", "description": "규정을 검색합니다", "parameters": {"type": "object"}},
    {"name": "list_chapters", "description": "장 목록을 반환합니다", "parameters": {"type": "object"}},
], ensure_ascii=False)

question = "연차를 다음 해로 넘길 수 있나요?"
history = "이전 대화: 근무 시간을 물어봤고 오전 10시부터라고 답했습니다. " * 6


def ntok(s):
    return len(tokenizer.encode(s))


budget = {
    "시스템 프롬프트": ntok(SYSTEM),
    "도구 정의": ntok(TOOLS_DEF),
    f"검색 문서 (전체 {len(chunks)}개)": sum(ntok(c) for c in chunks),
    "대화 히스토리": ntok(history),
    "사용자 질문": ntok(question),
}
total = sum(budget.values())

print(f"{'구성 요소':<24} {'토큰':>8} {'비중':>8}")
print("-" * 44)
for k, v in budget.items():
    print(f"{k:<24} {v:>8} {v/total:>7.0%}")
print("-" * 44)
print(f"{'합계':<24} {total:>8}")

구성 요소                          토큰       비중
--------------------------------------------
시스템 프롬프트                       63      9%
도구 정의                          58      8%
검색 문서 (전체 6개)                 445     61%
대화 히스토리                       151     21%
사용자 질문                         13      2%
--------------------------------------------
합계                            730

RAG 시스템에서는 대개 검색 문서가 가장 큰 비중을 차지합니다.

이제 빼기를 합니다. 문서 전체 대신 관련된 것만 넣으면 예산이 어떻게 바뀌는지 봅니다.

In [ ]:
# top-k를 줄였을 때 예산이 어떻게 변하는지 계산합니다
fixed = budget["시스템 프롬프트"] + budget["도구 정의"] + budget["사용자 질문"]

print(f"{'top-k':>6} {'문서 토큰':>10} {'전체 토큰':>10} {'절감률':>8}")
print("-" * 38)
for k in [len(chunks), 5, 3, 1]:
    doc_tokens = sum(ntok(c) for c in chunks[:k])
    tot = fixed + budget["대화 히스토리"] + doc_tokens
    print(f"{k:>6} {doc_tokens:>10} {tot:>10} {1 - tot/total:>7.0%}")

 top-k      문서 토큰      전체 토큰      절감률
--------------------------------------
     6        445        730      0%
     5        384        669      8%
     3        216        501     31%
     1         88        373     49%

top-k를 3으로 줄이면 전체 토큰이 크게 줄어듭니다. 그런데 **줄여도 답변 품질이 유지되는지**를 반드시 확인해야 합니다.

정답 문서가 4위에 있었다면 top-3으로 줄이는 순간 답을 못 합니다. 그래서 프라이빗 RAG 노드에서 **정답 청크가 상위 k개 안에 들어왔는지**(Hit@k)를 따로 쟀고 예산을 줄이는 결정도 그 숫자를 보고 내립니다.

In [ ]:
# top-k별 답변을 비교합니다
def rag_answer(question, k):
    ctx = "\n\n".join(chunks[:k])
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {question}"}]
    return chat(msgs, max_new_tokens=250)


for k in [len(chunks), 3, 1]:
    print(f"=== top-{k} ===")
    print(rag_answer("연차를 다음 해로 넘길 수 있나요?", k)[:200])
    print()

=== top-6 ===
네, 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 있습니다. 그러나 이월은 회사의 업무상 사유로 사용하지 못한 경우에 한하며, 이 때는 인사팀장의 승인을 받아야 합니다. 또한 이월 가능 일수는 최대 5일까지입니다. 이는 제7조에 근거합니다.

=== top-3 ===
해당 규정에는 연차를 다음 해로 넘길 수 있는지에 대한 내용이 명시되어 있지 않습니다. 따라서, 현재 제공된 규정에 따르면 확인되지 않습니다.

=== top-1 ===
해당 규정에서는 연차를 다음 해로 넘길 수 있는지에 대한 내용이 명시되어 있지 않습니다. 따라서, 이 문제는 규정에 근거하여 확인할 수 없습니다.

따라서, 답변은 다음과 같습니다:

확인되지 않습니다. (규정 제3조 참조)

연차 이월은 **제7조**이고 우리가 자른 순서에서 제7조는 네 번째 조각입니다. 그래서 top-3과 top-1에는 근거가 컨텍스트 윈도우 안에 아예 없습니다.

top-1 답변 끝의 "(규정 제3조 참조)"도 보십시오. 근거가 없다고 말하면서 조항 번호를 붙였고 그 조항은 근무 시간 조항입니다. 형식은 맞고 내용이 틀린 답변이고, Step 4에서 만들 규칙 센서는 이런 답을 통과시킵니다.

모델이 못 맞히는 것이 아니라 **줄 수 있는 근거를 우리가 빼 버린 것**입니다. 토큰을 크게 아꼈어도 답을 못 하면 아낀 것이 아니므로 예산을 줄이는 결정은 **정답률과 함께** 봅니다.

## Step 3 : 중간 소실 재현

이론 레슨에서 본 중간 소실을 직접 재현합니다. 정답이 담긴 문서를 컨텍스트의 앞, 중간, 뒤에 각각 두고 정답률을 비교합니다.

**얼마나 채우는지가 실험의 성패를 가릅니다.** 컨텍스트 윈도우의 5%만 채우면 모델은 중간에 있는 문서도 잘 찾아내 세 위치가 모두 100%로 나오고 "중간 소실은 과장된 이야기였네"라고 잘못 결론 내리게 됩니다. 컨텍스트 윈도우의 30%를 넘게 채우고 방해 문서도 정답 문서와 비슷한 주제로 만들어 정답 문서가 눈에 띄지 않게 해야 위치에 따른 차이가 보입니다. 그래도 모델에 따라서는 차이가 작게 나올 수 있습니다.

In [ ]:
# GQA(그룹 질의 어텐션)는 여러 질의 헤드가 키·값 헤드 하나를 나눠 쓰는 구조이고 Qwen2.5가 이 구조입니다.
# 플래시 어텐션 커널이 없는 GPU(T4 등)에서는 GQA 경로가 느리고 메모리를 많이 쓰는 기본 커널로 떨어지므로 여기서만 끕니다.
# 키·값을 헤드 수만큼 복제해 메모리 효율 커널로 보냅니다. 계산 결과는 같습니다
import transformers.integrations.sdpa_attention as sdpa_mod

if torch.cuda.get_device_capability()[0] < 8:
    sdpa_mod.use_gqa_in_sdpa = lambda *args, **kwargs: False

In [ ]:
# 채우기용 문서를 만듭니다. 정답 문서와 비슷한 주제(휴가 운영 세칙)이지만 정답은 들어 있지 않습니다
# 정답과 무관한 문서로 채우면 모델이 정답 문서를 쉽게 골라내 세 위치가 모두 100%로 나옵니다.
# 비슷한 문서 사이에 묻혀 있어야 위치에 따른 차이가 드러납니다
N_FILLER = 100          # 시간이 부족하면 60으로 줄이세요. 40 미만에서는 세 위치가 모두 100%로 나오기 쉽습니다

# 컨텍스트 윈도우 크기를 코드에 박지 않습니다. 모델이 알려주는 값을 씁니다.
# 모델을 바꾸면 이 숫자도 따라 바뀌어야 하는데, 박아두면 따라오지 않습니다
if BACKEND == "local":
    WINDOW = _model.config.max_position_embeddings
else:
    WINDOW = getattr(tokenizer, "model_max_length", 32768)
print(f"컨텍스트 윈도우 {WINDOW:,} 토큰 (모델 설정에서 읽었습니다)")

LEAVE_KINDS = ["병가", "경조 휴가", "공가", "보상 휴가", "장기근속 휴가", "교육 휴가", "출산 전후 휴가", "가족 돌봄 휴가"]
APPROVERS = ["부서장", "본부장", "대표이사", "총무팀장"]   # 정답 문서의 승인권자 '인사팀장'은 넣지 않습니다
DAYS = [2, 3, 7, 10]                                       # 정답 문서의 '5일'은 넣지 않습니다

filler = [
    f"## 부칙 제{i}항 ({LEAVE_KINDS[i % len(LEAVE_KINDS)]}의 운영)\n"
    f"{LEAVE_KINDS[i % len(LEAVE_KINDS)]}는 당해 연도 안에 사용함을 원칙으로 하되, "
    f"부득이한 사유가 있는 경우 {APPROVERS[i % len(APPROVERS)]}의 승인을 받아 "
    f"최대 {DAYS[i % len(DAYS)]}일까지 다음 분기 안에 나누어 사용할 수 있다. "
    f"신청은 사용 예정일 {i % 4 + 1}일 전까지 하며, 증빙 서류는 복귀 후 {i % 3 + 6}일 이내에 제출한다. "
    f"기한 안에 사용하지 않은 일수는 소멸하며 금전으로 보상하지 않는다."
    for i in range(1, N_FILLER + 1)
]

gold_doc = [c for c in chunks if "이월" in c][0]
filler_tok = sum(ntok(f) for f in filler)
print(f"정답 문서 : {gold_doc[:40]}...")
print(f"채우기 문서 {len(filler)}개, 총 {filler_tok:,} 토큰")
print(f"컨텍스트 윈도우 대비 {filler_tok/WINDOW:.0%} 사용  (컨텍스트 윈도우 {WINDOW:,} 토큰)")

컨텍스트 윈도우 32,768 토큰 (모델 설정에서 읽었습니다)
정답 문서 : ## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는...
채우기 문서 100개, 총 13,241 토큰
컨텍스트 윈도우 대비 40% 사용  (컨텍스트 윈도우 32,768 토큰)

In [ ]:
# 정답 문서의 위치를 바꿔가며 정답률을 측정합니다
# 정답 판정은 "인사팀장"과 "5일"이 둘 다 답변에 있는지로 합니다. 둘 다 정답 문서에만 등장합니다
# 질문은 정답 문서의 표현("연차", "이월")을 그대로 쓰지 않습니다. 표현이 겹치면 모델이 글자만 맞춰 찾아내므로
# 위치에 따른 차이가 잘 안 보입니다
# 이 실험에서는 샘플링을 켭니다. 그리디(temperature=0)는 같은 입력에 항상 같은 답을 주므로
# 몇 번을 돌려도 0%와 100% 사이에 아무 값도 나오지 않습니다
PROBE_TEMP = 0.7
PROBE_TRIALS = 5
PROBE_QUESTION = "작년에 다 쓰지 못한 휴가를 올해 쓸 수 있는 조건과 승인권자는?"


def is_hit(ans):
    return "인사팀장" in ans and "5일" in ans


def probe(position, trials=PROBE_TRIALS):
    """position: 'front' | 'middle' | 'back'"""
    if position == "front":
        ctx = [gold_doc] + filler
    elif position == "middle":
        mid = len(filler) // 2
        ctx = filler[:mid] + [gold_doc] + filler[mid:]
    else:
        ctx = filler + [gold_doc]

    hits = 0
    for _ in range(trials):
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user",
                 "content": "[규정]\n" + "\n\n".join(ctx) + f"\n\n질문: {PROBE_QUESTION}"}]
        ans = chat(msgs, max_new_tokens=200, temperature=PROBE_TEMP)
        hits += int(is_hit(ans))
    return hits / trials


print(f"{'위치':>8} {'정답률':>8}   (각 {PROBE_TRIALS}회 시행)")
print("-" * 32)
for pos in ["front", "middle", "back"]:
    print(f"{pos:>8} {probe(pos):>7.0%}")

      위치      정답률   (각 5회 시행)
--------------------------------
   front      0%
  middle    100%
    back     60%

위치에 따라 정답률이 크게 갈렸다면 실험은 성공입니다. 다만 어느 자리가 약한지는 모델·길이·방해 문서에 따라 달라서 문헌의 U자(가운데가 가장 약함)와 다르게 나올 수 있습니다. 위 실행에서는 앞 0%, 중간 100%, 뒤 60%로 **앞자리가 가장 약했습니다.** 이 모델은 비슷한 조항 100개를 읽은 뒤에는 맨 앞에 둔 조항을 거의 쓰지 못했고, 뒤에 둔 조항도 다섯 번 중 두 번은 놓쳤습니다.

세 위치가 모두 100%라면 이 모델이 이 길이에서는 아직 정답을 잘 찾는 것이니 `N_FILLER`를 늘리거나 시행 횟수를 늘려 다시 재 보십시오. 가져갈 것은 "가운데가 약하다"는 공식이 아니라 **위치가 결과를 좌우하므로 내 모델과 내 문서로 재서 정해야 한다**는 사실입니다.

**측정 설정**

- **샘플링을 켰습니다.** 여기서 보려는 것은 "찾았다/못 찾았다"의 이분법이 아니라 **얼마나 잘 찾는가의 정도**이고 그 정도는 확률 분포에서만 드러납니다. 그리디로 5회를 돌리면 완전히 같은 답 5개가 나와 0%와 100% 사이가 비어 버립니다. 반대로 Step 4의 규약 준수 측정은 학생끼리 결과를 비교할 수 있어야 하므로 재현성을 위해 그리디로 합니다. **무엇을 재느냐에 따라 샘플링을 켜고 끕니다**  
- **시행이 5회입니다.** 5회로 추정한 비율의 해상도는 20%p라 40%와 60%의 차이는 우연일 수 있고 20%와 100%의 차이는 우연이기 어렵습니다. 더 정밀하게 보려면 시행을 늘려야 하고 그만큼 GPU 시간이 듭니다

실무에서는 **리랭킹으로 상위 문서를 가장 강한 자리에 두거나** **지시문을 컨텍스트 끝에 다시 반복**합니다. 어느 자리가 강한지는 위 측정으로 정합니다(이 실행에서는 앞이 아니라 중간과 뒤였습니다). 여기서는 지시 반복을 시험합니다.

In [ ]:
# 지시문을 끝에 반복하면 중간 소실이 완화되는지 확인합니다
def probe_with_reminder(trials=PROBE_TRIALS):
    mid = len(filler) // 2
    ctx = filler[:mid] + [gold_doc] + filler[mid:]
    hits = 0
    for _ in range(trials):
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user",
                 "content": "[규정]\n" + "\n\n".join(ctx) +
                            "\n\n위 규정 전체를 빠짐없이 검토하세요. 특히 쓰지 못한 휴가를 다음 해에 쓰는 조항을 찾으세요."
                            f"\n\n질문: {PROBE_QUESTION}"}]
        ans = chat(msgs, max_new_tokens=200, temperature=PROBE_TEMP)
        hits += int(is_hit(ans))
    return hits / trials


print(f"중간 배치 (지시 반복 없음) : {probe('middle'):.0%}")
print(f"중간 배치 (지시 반복 있음) : {probe_with_reminder():.0%}")

중간 배치 (지시 반복 없음) : 80%
중간 배치 (지시 반복 있음) : 40%

지시 반복의 효과도 실행마다 다릅니다. 위 실행에서는 반복 없음 80%, 반복 있음 40%로 오히려 떨어졌습니다. 두 가지를 같이 보십시오. 첫째, 같은 중간 배치가 바로 앞 셀에서는 100%, 이 셀에서는 80%로 나왔습니다. 5회 시행의 해상도(20%p) 안에서 값이 흔들린다는 뜻이라 80%와 40%의 차이도 우연 범위에 걸칩니다. 둘째, 이 실행에서 중간 배치는 이미 거의 다 맞혀서 지시 반복으로 올릴 여지가 없었습니다. 지시 반복은 **약한 자리**(이 실행에서는 앞)에 정답 문서를 두고 시험해야 효과가 보입니다. `probe_with_reminder`의 배치를 앞으로 바꾸고 `PROBE_TRIALS`를 10으로 늘려 다시 재 보십시오.

## Step 4 : 하네스 만들기, 가이드와 센서

이론 레슨에서 하네스는 **가이드**와 **센서** 두 축으로 되어 있고 우리가 지금까지 센서가 약했다고 했습니다. 같은 모델에 센서만 붙여 품질이 어떻게 달라지는지 잽니다.

태스크는 **사내 규정 안내 답변 작성**이고 조직에 답변 작성 규약이 있다고 가정합니다.

In [ ]:
# 가이드 : 규약 문서
GUIDE = """[사내 답변 작성 규약]
1. 답변은 3문장 이내로 작성한다.
2. 반드시 근거 조항 번호를 '제N조' 형식으로 표기한다. 단, 규정에서 확인되지 않는 경우에는 표기하지 않는다.
3. 조건이 있는 규정은 조건을 빠뜨리지 않는다.
4. 추측이나 일반론을 쓰지 않는다. 규정에 없으면 '규정에서 확인되지 않습니다'라고 답한다.
5. 존댓말을 사용한다."""

In [ ]:
# 센서 : 규약 준수를 기계적으로 검사합니다. 모델이 아니라 규칙 기반입니다
# 용어 대응 : 여기서 말하는 '규칙 기반 센서'가 이론 레슨 4.5절의 '계산형 센서'입니다.
#            뒤의 트랙 2에 나오는 '판정 모델 센서'가 이론 레슨의 '추론형 센서'입니다
def check_answer(ans: str):
    issues = []
    sentences = [s for s in re.split(r"[.!?]\s*", ans.strip()) if s]
    if len(sentences) > 3:
        issues.append(f"규약1 위반: {len(sentences)}문장 (3문장 이내여야 함)")
    # 규약4에 따라 "확인되지 않습니다"로 답한 경우에는 인용할 조항이 없습니다.
    # 이 예외를 빼먹으면 규정에 없는 질문은 몇 번을 다시 써도 통과하지 못합니다
    unanswerable = bool(re.search(r"확인되지\s?않", ans))
    if not unanswerable and not re.search(r"제\s?\d+\s?조", ans):
        issues.append("규약2 위반: 근거 조항 번호 없음")
    # 존댓말은 '~니다'와 '~세요/~시오'로 봅니다. '습니다'와 '입니다'만 찾으면
    # '가능합니다', '바랍니다', '주십시오'가 전부 반말로 잡힙니다
    if not re.search(r"(니다|세요|시오)", ans):
        issues.append("규약5 위반: 존댓말 아님")
    return {"pass": len(issues) == 0, "issues": issues}


# 센서 동작 확인
print(check_answer("연차는 이월할 수 없다."))
print(check_answer("원칙적으로 이월이 불가능합니다. 다만 업무상 사유가 있으면 인사팀장 승인으로 최대 5일까지 가능합니다. 근거는 제7조입니다."))
print(check_answer("해당 내용은 규정에서 확인되지 않습니다."))   # 조항 인용 면제가 동작하는지

{'pass': False, 'issues': ['규약2 위반: 근거 조항 번호 없음', '규약5 위반: 존댓말 아님']}
{'pass': True, 'issues': []}
{'pass': True, 'issues': []}

센서를 만들 때는 **규약끼리 충돌하는 것을 놓치는** 실수가 가장 흔합니다.

규약2는 조항 번호를 반드시 쓰라고 하고 규약4는 규정에 없으면 확인되지 않는다고 답하라고 합니다. 그런데 규정에 없는 질문에는 인용할 조항이 없습니다. 센서가 조항 인용을 무조건 요구하면 **규약4를 지킨 올바른 답이 규약2 위반으로 잡혀** 몇 번을 다시 써도 통과하지 못합니다.

이 실수는 조건 A·B·C에서 잴 준수율의 상한을 조용히 내립니다. 평가 세트 5개 중 1개가 영구 실패라면 준수율은 아무리 잘해도 80%에서 멈춥니다. 그 80%를 보고 우리는 "모델의 한계"라고 결론 내리지만 실제로는 센서의 버그입니다. **측정 도구의 버그는 모델의 한계로 위장됩니다.**

숫자로 봅니다. 예외를 빼먹은 센서를 하나 만들어 완벽한 답변 다섯 개를 두 센서에 똑같이 넣습니다. GPU를 쓰지 않으니 금방 끝납니다.

In [ ]:
# 예외를 빼먹은 센서. 규약2를 무조건 요구합니다
def check_answer_buggy(ans: str):
    issues = []
    sentences = [s for s in re.split(r"[.!?]\s*", ans.strip()) if s]
    if len(sentences) > 3:
        issues.append(f"규약1 위반: {len(sentences)}문장")
    if not re.search(r"제\s?\d+\s?조", ans):          # 예외 없음
        issues.append("규약2 위반: 근거 조항 번호 없음")
    if not re.search(r"(니다|세요|시오)", ans):
        issues.append("규약5 위반: 존댓말 아님")
    return {"pass": len(issues) == 0, "issues": issues}


# 규약을 완벽하게 지킨 답변 다섯 개입니다. 사람이 채점하면 5/5입니다
perfect = [
    "원칙적으로 이월할 수 없습니다. 업무상 사유가 있으면 인사팀장 승인으로 5일까지 가능합니다. 근거는 제7조입니다.",
    "재택근무는 근무일 전일 오후 6시까지 신청해야 합니다. 근거는 제5조입니다.",
    "배우자 출산 휴가는 10일입니다. 근거는 제8조입니다.",
    "코어 타임은 오전 11시부터입니다. 근거는 제4조입니다.",
    "해당 내용은 규정에서 확인되지 않습니다.",           # 규약4를 지킨 답변
]

for name, sensor in [("예외 있음 (고친 센서)", check_answer),
                     ("예외 없음 (버그 센서)", check_answer_buggy)]:
    hit = sum(sensor(a)["pass"] for a in perfect)
    print(f"{name:<22} {hit}/{len(perfect)}  (상한 {hit/len(perfect):.0%})")

예외 있음 (고친 센서)          5/5  (상한 100%)
예외 없음 (버그 센서)          4/5  (상한 80%)

버그 센서는 완벽한 답변 다섯 개에 80%를 줍니다. **모델은 이 측정에 참여조차 하지 않았는데** 숫자는 이미 80%에서 막혀 있습니다.

이 상태로 조건 A·B·C를 재면 어느 조건도 100%에 닿지 못하고 우리는 "7B 모델의 한계"라고 적게 됩니다. 이론 레슨 4.5절의 "**측정 도구의 버그는 측정 대상의 한계처럼 보입니다**"가 이 장면입니다.

**그래서 센서를 만들면 센서부터 시험합니다.** 통과해야 할 답변과 떨어져야 할 답변을 손으로 몇 개 만들어 넣어 봅니다. 완벽한 답변 다섯 개를 두 센서에 넣은 셀이 그 작업이고 1분이면 됩니다.

In [ ]:
# 평가 세트
qa_set = [
    ("연차를 다음 해로 넘길 수 있나요?", "이월"),
    ("재택근무는 언제까지 신청해야 하나요?", "전일"),
    ("배우자가 출산하면 휴가가 며칠인가요?", "10"),
    ("코어 타임은 몇 시부터인가요?", "11"),
    ("점심시간은 몇 시인가요?", None),          # 규정에 없는 질문
]


def answer_once(q, guide=None):
    ctx = "\n\n".join(chunks)
    sys = SYSTEM + (f"\n\n{guide}" if guide else "")
    msgs = [{"role": "system", "content": sys},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
    # 규약 준수 측정은 그리디로 합니다. 같은 레슨을 돌린 사람끼리 결과가 같아야
    # 조건 A·B·C의 차이가 하네스 때문인지 우연 때문인지 구분할 수 있습니다
    return chat(msgs, max_new_tokens=250, temperature=0.0)

In [ ]:
# 조건 A : 가이드도 센서도 없음
print("=== A. 가이드 없음, 센서 없음 ===")
pass_a = 0
for q, _ in qa_set:
    a = answer_once(q)
    v = check_answer(a)
    pass_a += v["pass"]
    print(f"  [{'O' if v['pass'] else 'X'}] {q}")
    if not v["pass"]:
        print(f"       {v['issues']}")
print(f"규약 준수율 : {pass_a}/{len(qa_set)}")

=== A. 가이드 없음, 센서 없음 ===
  [X] 연차를 다음 해로 넘길 수 있나요?
       ['규약1 위반: 4문장 (3문장 이내여야 함)']
  [O] 재택근무는 언제까지 신청해야 하나요?
  [O] 배우자가 출산하면 휴가가 며칠인가요?
  [O] 코어 타임은 몇 시부터인가요?
  [X] 점심시간은 몇 시인가요?
       ['규약2 위반: 근거 조항 번호 없음']
규약 준수율 : 3/5

In [ ]:
# 조건 B : 가이드만 제공
print("=== B. 가이드 있음, 센서 없음 ===")
pass_b = 0
for q, _ in qa_set:
    a = answer_once(q, guide=GUIDE)
    v = check_answer(a)
    pass_b += v["pass"]
    print(f"  [{'O' if v['pass'] else 'X'}] {q}")
print(f"규약 준수율 : {pass_b}/{len(qa_set)}")

=== B. 가이드 있음, 센서 없음 ===
  [O] 연차를 다음 해로 넘길 수 있나요?
  [O] 재택근무는 언제까지 신청해야 하나요?
  [O] 배우자가 출산하면 휴가가 며칠인가요?
  [O] 코어 타임은 몇 시부터인가요?
  [O] 점심시간은 몇 시인가요?
규약 준수율 : 5/5

In [ ]:
# 실습 : 조건 C를 구현하세요. 가이드 + 센서 + 피드백 재주입
# 센서가 위반을 잡으면 위반 내용을 알려주고 다시 쓰게 합니다. 최대 3회.

def answer_with_sensor(q, max_retry=3):
    ctx = "\n\n".join(chunks)
    sys = SYSTEM + "\n\n" + GUIDE
    msgs = [{"role": "system", "content": sys},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]

    for attempt in range(max_retry):
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        # 여기에 검사와 피드백 재주입 로직을 작성하세요
        pass

    return ans, False, max_retry

<details>
<summary>정답 보기</summary>

```python
def answer_with_sensor(q, max_retry=3):
    ctx = "\n\n".join(chunks)
    sys = SYSTEM + "\n\n" + GUIDE
    msgs = [{"role": "system", "content": sys},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]

    for attempt in range(max_retry):
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        v = check_answer(ans)
        if v["pass"]:
            return ans, True, attempt + 1

        # 피드백 재주입 : 위반 내용을 구체적으로 알려줍니다
        msgs.append({"role": "assistant", "content": ans})
        msgs.append({"role": "user",
                     "content": "작성한 답변이 규약을 위반했습니다.\n"
                                + "\n".join(f"- {i}" for i in v["issues"])
                                + "\n위반 사항을 고쳐 답변을 다시 작성하세요."})

    return ans, False, max_retry
```

**위반 내용을 구체적으로 알려 주십시오.** "다시 쓰세요"만 말하면 같은 실수를 반복합니다. "3문장 이내여야 하는데 5문장입니다"처럼 무엇이 어떻게 틀렸는지 전달해야 다음 시도가 나아집니다.

**실행 순서** : 정답 코드를 실습 셀에 붙여 다시 실행한 다음 조건 C 셀을 실행하십시오. 순서가 바뀌면 `pass`만 있는 함수가 그대로 호출되어 **오류 없이** 준수율 0%와 시도 3회가 찍힙니다. 실패가 오류로 오지 않는다는 것을 여기서도 확인하십시오.

</details>

In [ ]:
print("=== C. 가이드 + 센서 + 피드백 재주입 ===")
pass_c, total_attempts = 0, 0
for q, _ in qa_set:
    a, ok, tries = answer_with_sensor(q)
    pass_c += ok
    total_attempts += tries
    print(f"  [{'O' if ok else 'X'}] {q}  (시도 {tries}회)")
print(f"규약 준수율 : {pass_c}/{len(qa_set)}")
print(f"평균 시도 횟수 : {total_attempts/len(qa_set):.1f}")

=== C. 가이드 + 센서 + 피드백 재주입 ===
  [O] 연차를 다음 해로 넘길 수 있나요?  (시도 1회)
  [O] 재택근무는 언제까지 신청해야 하나요?  (시도 1회)
  [O] 배우자가 출산하면 휴가가 며칠인가요?  (시도 1회)
  [O] 코어 타임은 몇 시부터인가요?  (시도 1회)
  [O] 점심시간은 몇 시인가요?  (시도 1회)
규약 준수율 : 5/5
평균 시도 횟수 : 1.0

In [ ]:
print(f"{'조건':<28} {'준수율':>8}")
print("-" * 40)
print(f"{'A. 가이드 없음, 센서 없음':<28} {pass_a/len(qa_set):>7.0%}")
print(f"{'B. 가이드만':<28} {pass_b/len(qa_set):>7.0%}")
print(f"{'C. 가이드 + 센서':<28} {pass_c/len(qa_set):>7.0%}")

조건                                준수율
----------------------------------------
A. 가이드 없음, 센서 없음                 60%
B. 가이드만                         100%
C. 가이드 + 센서                     100%

**모델은 세 조건에서 완전히 동일합니다.** 같은 가중치, 같은 4bit 양자화입니다. 달라진 것은 하네스뿐입니다.

이론 레슨에서 인용한 "괜찮은 모델에 훌륭한 하네스를 붙인 쪽이, 훌륭한 모델에 나쁜 하네스를 붙인 쪽을 이깁니다"를 숫자로 본 것이고 프라이빗 환경의 7B 모델로 쓸 만한 시스템을 만들 수 있다는 근거입니다.

A와 B의 차이가 가이드의 효과입니다. B와 C가 같은 100%로 나와도 이상한 것이 아닙니다. 다섯 문항이 가이드만으로 모두 통과했다면 센서가 고칠 것이 없어 C도 100%이고 평균 시도 횟수는 1.0입니다. 센서의 값어치는 가이드만으로 안 되는 문항이 있을 때 드러나고, 가이드만 있으면 지켰는지 확인조차 못 한다는 점은 변하지 않습니다. Step 5에서 규약을 강화해 한 번에 통과하지 못하는 장면을 만들고 그때 센서와 재시도가 무엇을 하는지 봅니다.

## Step 5 : 루프 엔지니어링

Step 4의 재시도는 루프의 최소 형태였습니다. 이제 루프 자체를 설계합니다.

#### 5.1 검증 루프의 수렴 곡선

루프를 무한정 돌릴 수는 없으니 **몇 번까지 돌려야 개선되는지** 알아야 합니다. 규약을 강화해 한 번에 통과하기 어렵게 만든 태스크로 잽니다.

In [ ]:
STRICT_GUIDE = GUIDE + """
6. 답변의 첫 문장은 반드시 '규정에 따르면'으로 시작한다.
7. 숫자는 아라비아 숫자로만 표기한다. (예: '다섯 일'이 아니라 '5일')
8. 답변 끝에 '(근거: 제N조)' 형식으로 마무리한다. 규정에서 확인되지 않는 경우에는 생략한다."""


TAIL = r"\(근거\s?:\s?제\s?\d+\s?조\)"


def check_strict(ans: str):
    issues = []
    # check_answer와 같은 예외를 여기에도 둡니다. 센서를 강화할 때
    # 앞에서 고친 예외를 빼먹는 것이 실제로 가장 잦은 실수입니다
    unanswerable = bool(re.search(r"확인되지\s?않", ans))
    # 규약8이 요구하는 '(근거: 제N조)'는 문장이 아닙니다. 빼고 셉니다.
    # 빼지 않으면 규약1(3문장 이내)과 규약8이 서로를 위반하게 만듭니다
    body = re.sub(TAIL + r"\s*$", "", ans.strip())
    sentences = [s for s in re.split(r"[.!?]\s*", body) if s]
    if len(sentences) > 3:
        issues.append(f"규약1 위반: {len(sentences)}문장")
    if not unanswerable and not re.search(r"제\s?\d+\s?조", ans):
        issues.append("규약2 위반: 조항 번호 없음")
    if not ans.strip().startswith("규정에 따르면"):
        issues.append("규약6 위반: 첫 문장이 '규정에 따르면'으로 시작하지 않음")
    if re.search(r"(하나|둘|셋|넷|다섯|열)\s?일", ans):
        issues.append("규약7 위반: 숫자를 한글로 표기")
    # '끝에'라고 했으므로 끝에서 찾습니다. 아무 데나 있어도 통과시키면
    # 규약8이 사실상 규약2와 같은 검사가 됩니다
    if not unanswerable and not re.search(TAIL + r"\s*$", ans.strip()):
        issues.append("규약8 위반: 끝에 '(근거: 제N조)' 형식 없음")
    return {"pass": len(issues) == 0, "issues": issues}

In [ ]:
# 반복 횟수별 통과율을 측정해 수렴 곡선을 만듭니다
# 각 질문에 대해 최대 n회까지 루프를 돌렸을 때의 통과율을 기록합니다

def loop_until_pass(q, max_iter, checker, guide):
    ctx = "\n\n".join(chunks)
    msgs = [{"role": "system", "content": SYSTEM + "\n\n" + guide},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
    for i in range(max_iter):
        # 피드백이 메시지에 쌓이므로 입력이 매 회 달라집니다. 그리디여도 답이 바뀝니다
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        v = checker(ans)
        if v["pass"]:
            return True, i + 1
        msgs.append({"role": "assistant", "content": ans})
        msgs.append({"role": "user", "content": "규약 위반:\n" + "\n".join(v["issues"])
                                                + "\n고쳐서 다시 작성하세요."})
    return False, max_iter


print(f"{'최대 반복':>8} {'통과율':>8} {'평균 시도':>10}")
print("-" * 30)
for max_iter in [1, 2, 3, 5]:
    results = [loop_until_pass(q, max_iter, check_strict, STRICT_GUIDE) for q, _ in qa_set]
    rate = sum(r[0] for r in results) / len(results)
    avg = sum(r[1] for r in results) / len(results)
    print(f"{max_iter:>8} {rate:>7.0%} {avg:>10.1f}")

   최대 반복      통과율      평균 시도
------------------------------
       1     80%        1.0
       2    100%        1.2
       3    100%        1.2
       5    100%        1.2

1회에서 2회로 갈 때 크게 오르고 3회 이후에는 완만해지는 형태가 일반적입니다.

**정체 지점을 찾는 것이 루프 엔지니어링의 실무 작업입니다.** 5회를 돌려도 안 되는 질문은 10회를 돌려도 안 됩니다. 그런 경우는 루프를 늘리는 대신 가이드를 고치거나 태스크를 쪼개거나 사람에게 넘겨야 합니다.

다만 정체를 봤을 때는 모델이 규약을 못 지키는 것인지 센서가 지킬 수 없는 것을 요구하는 것인지 **원인을 나눠서 확인해야** 합니다. `check_strict`에 규약2·규약8 예외를 넣지 않았다면 규정에 없는 질문 한 개가 영구 실패로 남아 통과율이 80%에서 멈췄을 것이고 우리는 그것을 모델의 한계로 읽었을 것입니다. 정체가 보이면 **끝까지 실패한 질문을 직접 꺼내 답변과 위반 목록을 읽고** 위반 목록이 말이 되지 않으면 센서를 고치십시오.

이제 **정지 조건을 세어 봅니다.** 방금 만든 `loop_until_pass`를 이론 레슨 5.3절의 정지 조건 넷, **성공, 한계, 정체, 위임**에 대 봅니다.

![image.png](https://resources-public-prd.modulabs.co.kr/home-section/main-banner/044979f8-4df4-4753-a3c6-05e668fdbf6c.png)

넷 중 둘만 있어서 이 루프는 끝까지 통과하지 못하는 질문이 있으면 **같은 위반을 상한까지 반복**합니다. 수렴 곡선에서 최대 반복을 늘렸는데 통과율은 그대로이고 평균 시도만 올라갔다면 그 증가분이 이 낭비입니다. 다섯 문항이 모두 2회 안에 통과한 위 결과에서는 낭비가 보이지 않지만, 운영에서는 끝까지 실패하는 요청이 반드시 섞입니다.

정체 종료를 붙입니다. **직전 회차의 위반 목록과 이번 회차의 위반 목록이 같으면 멈춥니다.** 위반 목록은 문자열 리스트라 그대로 비교하면 됩니다.

위임은 조건이라기보다 **출구**입니다. 정체로 멈춘 요청을 사람에게 올리면 그것이 위임이므로 정체 종료를 만들면 위임 지점도 함께 생깁니다. 그래서 `loop_with_stall` 실습에서는 종료 이유를 문자열로 남깁니다.

In [ ]:
# 실습 : loop_until_pass에 정체 종료를 추가하세요
# 요구사항
#  1) 직전 회차의 위반 목록과 이번 회차의 위반 목록이 같으면 즉시 중단
#  2) 종료 이유를 "성공" / "정체" / "한계" 중 하나의 문자열로 함께 반환
#     (정체로 멈춘 것은 사람에게 넘길 목록이 됩니다. 이것이 네 번째 조건인 위임의 출구입니다)
# 힌트 : 위반 목록은 문자열 리스트입니다. 리스트끼리 == 로 비교하면 됩니다

def loop_with_stall(q, max_iter, checker, guide):
    ctx = "\n\n".join(chunks)
    msgs = [{"role": "system", "content": SYSTEM + "\n\n" + guide},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
    prev_issues = None

    for i in range(max_iter):
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        # 여기에 검사, 정체 판정, 피드백 재주입을 작성하세요
        pass

    return False, max_iter, "한계"

<details>
<summary>정답 보기</summary>

```python
def loop_with_stall(q, max_iter, checker, guide):
    ctx = "\n\n".join(chunks)
    msgs = [{"role": "system", "content": SYSTEM + "\n\n" + guide},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
    prev_issues = None

    for i in range(max_iter):
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        v = checker(ans)
        if v["pass"]:
            return True, i + 1, "성공"

        # 정체 : 구체적으로 알려줬는데 같은 위반이 그대로 돌아왔습니다
        if v["issues"] == prev_issues:
            return False, i + 1, "정체"
        prev_issues = v["issues"]

        msgs.append({"role": "assistant", "content": ans})
        msgs.append({"role": "user", "content": "규약 위반:\n" + "\n".join(v["issues"])
                                                + "\n고쳐서 다시 작성하세요."})

    return False, max_iter, "한계"
```

비교 대상은 답변 문장이 아니라 **위반 목록**입니다. 모델은 매번 문장을 조금씩 바꿔 쓰므로 답변을 비교하면 거의 항상 달라 정체를 영영 못 잡습니다. 우리는 "표현이 바뀌었는가" 대신 "**지적이 해소되었는가**"를 봅니다.

여유를 주고 싶으면 두 번 연속 같을 때 멈추게 하십시오. 카운터 하나만 더 두면 됩니다.

**실행 순서** : 정답 코드를 실습 셀에 붙여 다시 실행한 다음 두 루프 비교 셀을 실행하십시오.

</details>

In [ ]:
# 두 루프를 같은 조건에서 비교합니다. 통과율은 유지하면서 호출을 줄였는지 봅니다
from collections import Counter

MAX_ITER = 5
base = [loop_until_pass(q, MAX_ITER, check_strict, STRICT_GUIDE) for q, _ in qa_set]
stall = [loop_with_stall(q, MAX_ITER, check_strict, STRICT_GUIDE) for q, _ in qa_set]

print(f"{'루프':<22} {'통과율':>8} {'총 호출':>8}")
print("-" * 42)
print(f"{'정체 종료 없음':<22} {sum(r[0] for r in base)/len(base):>7.0%} {sum(r[1] for r in base):>8}")
print(f"{'정체 종료 있음':<22} {sum(r[0] for r in stall)/len(stall):>7.0%} {sum(r[1] for r in stall):>8}")

print("\n[종료 이유별 건수]")
for reason, n in Counter(r[2] for r in stall).items():
    print(f"  {reason} {n}건")

# 정체로 걸러진 질문은 버릴 로그가 아니라 고칠 작업 목록입니다
print("\n[정체로 멈춘 질문 = 사람에게 넘길 목록]")
for (q, _), r in zip(qa_set, stall):
    if r[2] == "정체":
        print(f"  {q}")

루프                          통과율     총 호출
------------------------------------------
정체 종료 없음                  100%        6
정체 종료 있음                  100%        6

[종료 이유별 건수]
  성공 5건

[정체로 멈춘 질문 = 사람에게 넘길 목록]

통과율은 그대로인데 총 호출이 줄었으면 성공입니다. **줄어든 호출은 어차피 같은 답을 받았을 호출이었습니다.** 다섯 문항이 모두 통과하는 조건에서는 두 루프의 호출 수가 같고 정체 목록도 비어 있습니다. 끝까지 실패하는 질문이 있을 때 차이가 납니다.

정체 목록이 비어 있지 않다면 그 질문들을 보십시오. 정체로 멈춘 질문들이 이제 **이름이 붙어 따로 모입니다.** 이론 레슨 5.4절의 표대로 원인 가운데 규약 모순과 무리한 센서는 우리가 고칠 수 있습니다.

**원인을 가리는 질문**

- 가이드의 규약끼리 모순되는가 (위반 지적 두 개가 서로 반대인지 봅니다)  
- 센서가 지킬 수 없는 것을 요구하는가 (위반 지적이 말이 되는지 봅니다)  
- 태스크가 이 모델의 능력 밖인가 (구체적으로 알려 줘도 못 고치는지 봅니다)

정체 종료가 없으면 이 셋이 전부 "모델이 부족하다"로 뭉뚱그려집니다. **정체 종료는 비용 장치이기 전에 진단 장치입니다.**

#### 5.2 컴팩션

루프가 길어지면 컨텍스트가 찹니다. Step 5.1의 루프에서 메시지가 어떻게 쌓이는지 봅니다.

In [ ]:
# 재시도가 반복되면 히스토리가 누적됩니다
ctx = "\n\n".join(chunks)
msgs = [{"role": "system", "content": SYSTEM + "\n\n" + STRICT_GUIDE},
        {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: 배우자 출산 휴가는 며칠인가요?"}]

for i in range(4):
    ans = chat(msgs, max_new_tokens=250, temperature=0.5)
    v = check_strict(ans)
    msgs.append({"role": "assistant", "content": ans})
    msgs.append({"role": "user", "content": "규약 위반:\n" + "\n".join(v["issues"])})
    tok = ntok(tokenizer.apply_chat_template(msgs, tokenize=False))
    print(f"{i+1}회차 후 : 메시지 {len(msgs)}개, 누적 {tok} 토큰")

1회차 후 : 메시지 4개, 누적 790 토큰
2회차 후 : 메시지 6개, 누적 836 토큰
3회차 후 : 메시지 8개, 누적 882 토큰
4회차 후 : 메시지 10개, 누적 928 토큰

In [ ]:
# 실습 : 컴팩션을 구현하세요
# 실패한 시도의 전문을 모두 남기는 대신, 위반 항목만 누적 목록으로 압축합니다
# 힌트 : 히스토리를 버리고 "지금까지 지적된 위반 목록"만 유지합니다

def loop_with_compaction(q, max_iter=4):
    ctx = "\n\n".join(chunks)
    base = [{"role": "system", "content": SYSTEM + "\n\n" + STRICT_GUIDE}]
    known_issues = []

    for i in range(max_iter):
        # 여기에 압축된 메시지 구성과 루프를 작성하세요
        pass

    return None

<details>
<summary>정답 보기</summary>

```python
def loop_with_compaction(q, max_iter=4):
    ctx = "\n\n".join(chunks)
    base = [{"role": "system", "content": SYSTEM + "\n\n" + STRICT_GUIDE}]
    known_issues = []

    for i in range(max_iter):
        note = ""
        if known_issues:
            note = ("\n\n[이전 시도에서 지적된 위반 사항. 반복하지 마세요]\n"
                    + "\n".join(f"- {x}" for x in dict.fromkeys(known_issues)))

        msgs = base + [{"role": "user",
                        "content": f"[규정]\n{ctx}\n\n질문: {q}{note}"}]
        tok = ntok(tokenizer.apply_chat_template(msgs, tokenize=False))

        ans = chat(msgs, max_new_tokens=250, temperature=0.3)
        v = check_strict(ans)
        print(f"{i+1}회차 : {tok} 토큰, pass={v['pass']}")

        if v["pass"]:
            return ans
        known_issues.extend(v["issues"])

    return ans
```

**히스토리 대신 상태만 누적합니다.** 실패한 답변 전문은 버리고 "무엇을 어겼는지"만 남깁니다.

컴팩션 덕에 토큰 수가 회차마다 거의 늘지 않아 루프를 오래 돌려도 컨텍스트가 넘치지 않습니다.

**실행 순서** : 정답 코드를 실습 셀에 붙여 다시 실행한 다음 `loop_with_compaction` 호출 셀을 실행하십시오. 붙이지 않고 실행하면 오류 없이 `None`만 찍힙니다.

</details>

In [ ]:
print(loop_with_compaction("배우자 출산 휴가는 며칠인가요?"))

1회차 : 744 토큰, pass=True
규정에 따르면, 배우자 출산 휴가는 1일을 부여합니다.(근거: 제8조)

출력의 답을 읽어 보십시오. 위 실행에서 모델은 배우자 출산 휴가가 1일이라고 답했고 센서는 통과시켰습니다. 규정은 10일이고, 1일은 자녀 결혼 휴가입니다. 첫 문장도, 조항 번호도, 끝의 (근거: 제N조) 형식도 다 맞아서 규칙 센서로는 잡을 수 없는 답이었습니다. 이론 레슨 4.5절에서 계산형 센서는 내용이 틀린 것을 못 잡는다고 한 것이 바로 이 경우이고, 트랙 2의 판정 모델 센서가 필요한 자리입니다.

이 실행에서는 1회차에 통과해서 컴팩션의 효과(회차가 늘어도 토큰이 거의 늘지 않음)는 보이지 않았습니다. 바로 앞 셀처럼 토큰이 쌓이는 모습(4회차에 790 → 928 토큰)과 비교하려면 질문을 더 어렵게 하거나 `STRICT_GUIDE`에 규약을 더해서 여러 회차가 돌게 해야 합니다.

## Step 6 : 선택 심화 트랙

다섯 트랙 중 **두 개를 선택**해 수행하고 공유합니다.

![표 : 트랙, 주제, 층위, 난이도](https://resources-public-prd.modulabs.co.kr/home-section/main-banner/a2878b02-5d1c-48d1-8e0b-b72299ab3f8f.jpg)

노트테이킹은 이론 레슨 3.4절에서 **빼기의 갈래 셋**(외부 기억)으로 나오니 2층입니다. 1.4절 표에서는 4층의 대표 작업으로도 나와서 트랙 1의 층위는 둘입니다. **한 번의 호출에서 컨텍스트 윈도우를 줄이는 쪽으로 보면 2층이고 여러 번의 작업 사이로 상태를 넘기는 쪽으로 보면 4층입니다.** 트랙 1을 고른다면 공유할 때 **둘 중 어느 쪽 문제를 풀었는지** 밝히십시오.

#### 트랙 1 : 노트테이킹으로 컨텍스트 외부화

컴팩션은 컨텍스트 안에서 압축했습니다. 노트테이킹은 아예 **밖으로 내보내** 파일에 기록하고 필요할 때만 읽습니다. 컨텍스트가 리셋되어도 작업을 이어 갈 수 있다는 점이 컴팩션과 다릅니다.

In [ ]:
# 실습 : 진행 상황을 파일에 기록하며 여러 질문을 순차 처리하는 루프를 만드세요
# 요구사항
#  1) 각 질문 처리 후 결과를 progress.json에 기록
#  2) 다음 질문 처리 시 이전 결과 요약만 컨텍스트에 넣기
#  3) 중간에 커널이 죽어도 파일을 읽어 이어갈 수 있는 구조

def load_notes(note_file):
    """기록 파일을 읽습니다. 깨져 있으면 없는 것으로 칩니다.
    중단된 시점에 파일이 반쯤 쓰였을 수 있어, 여기서 막히면 재시작 자체가 안 됩니다"""
    if not os.path.exists(note_file):
        return {}
    try:
        with open(note_file, encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError):
        print("기록 파일이 깨져 있어 처음부터 시작합니다")
        return {}


def process_with_notes(questions, note_file="progress.json"):
    notes = load_notes(note_file)

    for q in questions:
        if q in notes:
            print(f"[건너뜀] {q}")
            continue
        # 여기에 처리 및 기록 로직을 작성하세요
        pass

    return notes

<details>
<summary>정답 보기</summary>

```python
def process_with_notes(questions, note_file="progress.json"):
    notes = load_notes(note_file)

    for q in questions:
        if q in notes:
            print(f"[건너뜀] {q}")
            continue

        # 이전 결과는 요약만 컨텍스트에 넣습니다
        prev = "\n".join(f"- {k}: {v['summary']}" for k, v in list(notes.items())[-3:])
        note = f"\n\n[이전에 답한 항목]\n{prev}" if prev else ""

        ctx = "\n\n".join(chunks)
        msgs = [{"role": "system", "content": SYSTEM + "\n\n" + GUIDE},
                {"role": "user", "content": f"[규정]\n{ctx}{note}\n\n질문: {q}"}]
        ans = chat(msgs, max_new_tokens=250)

        notes[q] = {"answer": ans, "summary": ans[:60], "pass": check_answer(ans)["pass"]}
        with open(note_file, "w", encoding="utf-8") as f:
            json.dump(notes, f, ensure_ascii=False, indent=2)
        print(f"[완료] {q}")

    return notes
```

각 항목을 처리한 직후에 저장해야 중간에 끊겨도 잃는 것이 가장 적습니다.

**실행 순서** : 정답 코드를 실습 셀에 붙여 다시 실행한 다음 재실행 확인 셀을 실행하십시오. 이 셀은 같은 호출을 두 번 하고 두 번째에는 전부 건너뜁니다. 이것이 재시작 가능한 루프이고 아이펠톤처럼 여러 날에 걸친 작업에서 특히 쓸모 있습니다.

</details>

In [ ]:
# 이 셀을 통째로 다시 실행해도 결과는 같습니다. 기록이 파일에 남아 있기 때문입니다
qs = [q for q, _ in qa_set]

print("=== 1회차 ===")
process_with_notes(qs)

print("\n=== 2회차 (커널이 죽었다 살아난 셈) ===")
process_with_notes(qs)

=== 1회차 ===
[완료] 연차를 다음 해로 넘길 수 있나요?
[완료] 재택근무는 언제까지 신청해야 하나요?
[완료] 배우자가 출산하면 휴가가 며칠인가요?
[완료] 코어 타임은 몇 시부터인가요?
[완료] 점심시간은 몇 시인가요?

=== 2회차 (커널이 죽었다 살아난 셈) ===
[건너뜀] 연차를 다음 해로 넘길 수 있나요?
[건너뜀] 재택근무는 언제까지 신청해야 하나요?
[건너뜀] 배우자가 출산하면 휴가가 며칠인가요?
[건너뜀] 코어 타임은 몇 시부터인가요?
[건너뜀] 점심시간은 몇 시인가요?

{'연차를 다음 해로 넘길 수 있나요?': {'answer': '네, 가능합니다. 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 있으며, 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있습니다. (제7조)',
  'summary': '네, 가능합니다. 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 있으며, 회사의 업무상 사유',
  'pass': True},
 '재택근무는 언제까지 신청해야 하나요?': {'answer': '재택근무는 근무일 전일 오후 6시까지 부서장의 승인을 받아야 합니다. 제5조에 따릅니다.',
  'summary': '재택근무는 근무일 전일 오후 6시까지 부서장의 승인을 받아야 합니다. 제5조에 따릅니다.',
  'pass': True},
 '배우자가 출산하면 휴가가 며칠인가요?': {'answer': '배우자가 출산하면 휴가가 10일간 부여됩니다. 제8조에 따릅니다.',
  'summary': '배우자가 출산하면 휴가가 10일간 부여됩니다. 제8조에 따릅니다.',
  'pass': True},
 '코어 타임은 몇 시부터인가요?': {'answer': '코어 타임은 오전 11시부터 오후 4시까지입니다. 제4조에 따릅니다.',
  'summary': '코어 타임은 오전 11시부터 오후 4시까지입니다. 제4조에 따릅니다.',
  'pass': True},
 '점심시간은 몇 시인가요?': {'answer': '규정에서 확인되지 않습니다.',
  'summary': '규정에서 확인되지 않습니다.',
  'pass': True}}

#### 트랙 2 : 판정 모델 센서 vs 규칙 기반 센서

Step 4의 센서는 정규식 기반이었습니다. 형식은 잘 잡지만 **내용의 정확성**은 못 잡습니다.

판정 모델 센서를 만들어 두 방식을 비교합니다. 이론 레슨 4.5절의 용어로는 **규칙 기반 센서가 계산형, 판정 모델 센서가 추론형**입니다.

In [ ]:
JUDGE_SYSTEM = """당신은 답변을 검증하는 판정자입니다.
규정 원문과 답변을 비교해 아래를 판정하고 JSON으로만 답하세요.
1. 답변 내용이 규정과 일치하는가 (사실 정확성)
2. 조건이나 예외를 빠뜨리지 않았는가
형식 : {"pass": true/false, "issues": ["문제점"]}
issues 항목은 한국어로 작성하고, JSON 외의 텍스트는 출력하지 마세요."""


def judge_sensor(question, answer):
    ctx = "\n\n".join(chunks)
    r = chat([{"role": "system", "content": JUDGE_SYSTEM},
              {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {question}\n답변: {answer}"}],
             max_new_tokens=300, temperature=0.0)
    m = re.search(r"\{.*\}", r, re.S)
    # 판정 자체가 실패했을 때 무엇을 돌려줄지를 정해두어야 합니다.
    #   실패하면 통과시킨다 : pass=True. 편하지만 그 순간부터 센서가 조용히 꺼집니다
    #   실패하면 막는다     : pass=False. 시끄럽지만 못 본 채 나가는 답변이 없습니다
    # 형식이 깨진 판정은 통과가 아니라 실패로 둡니다
    try:
        return json.loads(m.group(0)) if m else {"pass": False, "issues": ["판정 형식 오류: JSON을 찾지 못함"]}
    except json.JSONDecodeError:
        return {"pass": False, "issues": ["판정 형식 오류: JSON 파싱 실패"]}

In [ ]:
# 실습 : 두 센서를 비교하세요
# 의도적으로 형식은 맞지만 내용이 틀린 답변, 내용은 맞지만 형식이 틀린 답변을 만들어
# 각 센서가 무엇을 잡고 무엇을 놓치는지 확인하세요

test_answers = [
    # (설명, 질문, 답변)
    ("형식 O, 내용 O", "연차 이월?",
     "규정에 따르면 원칙적으로 이월이 불가능합니다. 업무상 사유가 있으면 인사팀장 승인으로 최대 5일까지 가능합니다. (근거: 제7조)"),
    ("형식 O, 내용 X", "연차 이월?",
     "규정에 따르면 연차는 최대 10일까지 자유롭게 이월할 수 있습니다. 별도 승인은 필요하지 않습니다. (근거: 제7조)"),
    ("형식 X, 내용 O", "연차 이월?",
     "원칙적으로 불가능하지만 업무상 사유면 인사팀장 승인으로 5일까지 이월 가능"),
]

# 여기에 두 센서를 각각 적용해 결과를 표로 출력하세요

케이스                   규칙 센서      판정 센서
----------------------------------------
형식 O, 내용 O               통과         통과
형식 O, 내용 X               통과         탈락
형식 X, 내용 O               탈락         통과

<details>
<summary>정답 보기</summary>

```python
print(f"{'케이스':<16} {'규칙 센서':>10} {'판정 센서':>10}")
print("-" * 40)
for desc, q, a in test_answers:
    rule = check_answer(a)["pass"]
    llm = judge_sensor(q, a)["pass"]
    print(f"{desc:<16} {'통과' if rule else '탈락':>10} {'통과' if llm else '탈락':>10}")
```

**기대되는 결과**

- 형식 O 내용 X : 조항 번호가 있고 문장 수도 맞으니 규칙 센서는 **통과시키고** 판정 센서는 잡아냅니다  
- 형식 X 내용 O : 규칙 센서는 탈락시키고 판정 센서는 통과시킬 수 있습니다

그래서 **둘을 함께 씁니다.** 규칙 센서는 빠르고 결정적이지만 얕습니다. 판정 센서는 깊지만 느리고 불안정합니다. 실무에서는 규칙 센서로 먼저 거르고 통과한 것만 판정 센서에 넘겨 비용과 정확성을 함께 챙깁니다.

판정 센서가 7B 로컬 모델이라 **판정 자체가 틀릴 때도 있고** 같은 답변을 몇 번 넣어 보면 판정이 흔들리는 것도 보입니다. 이론 레슨 4.5절의 "센서가 거짓말할 때"가 이 자리입니다.

**센서를 검사하지 않고 센서로 모델을 검사하면 나온 숫자가 무엇을 잰 것인지 아무도 모릅니다.** 판정 센서를 쓸 때는 판정 결과의 표본을 사람이 직접 읽어 확인하는 절차를 함께 두십시오.

**실행 순서** : 정답 코드를 실습 셀 끝에 붙여 다시 실행하십시오. `test_answers`가 정의된 뒤에 실행되어야 합니다.

</details>

#### 트랙 3 : 컨텍스트 압축 실험

빼기의 네 갈래 중 **압축**을 실험합니다. 원문 대신 요약을 넣으면 토큰은 줄지만 정보가 빠집니다. 그 균형점을 찾습니다.

In [ ]:
# 실습 : 문서를 압축률별로 요약해 넣고, 토큰 수와 답변 정확도를 함께 측정하세요
# 압축은 로컬 모델로 수행합니다

def compress(text, target_ratio):
    target = int(len(text) * target_ratio)
    r = chat([{"role": "system", "content":
               f"주어진 규정 조항을 약 {target}자로 요약하세요. "
               "조건, 예외, 숫자, 승인권자는 반드시 유지하세요. 요약문만 출력하세요. "
               "요약은 반드시 한국어로만 작성하세요."},
              {"role": "user", "content": text}], max_new_tokens=300, temperature=0.0)
    return r.strip()


# 여기에 압축률 [1.0, 0.6, 0.4]별로 토큰 수와 정답률을 측정하는 코드를 작성하세요

     압축률      문서 토큰      정답률
------------------------------
     1.0        445    100%
     0.6        261    100%
     0.4        254    100%

<details>
<summary>정답 보기</summary>

```python
print(f"{'압축률':>8} {'문서 토큰':>10} {'정답률':>8}")
print("-" * 30)

for ratio in [1.0, 0.6, 0.4]:
    if ratio == 1.0:
        docs = chunks
    else:
        docs = [compress(c, ratio) for c in chunks]

    tok = sum(ntok(d) for d in docs)
    hits = 0
    for q, key in qa_set:
        if key is None:
            continue
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": "[규정]\n" + "\n\n".join(docs) + f"\n\n질문: {q}"}]
        ans = chat(msgs, max_new_tokens=200, temperature=0.0)
        hits += int(key in ans)
    n = sum(1 for _, k in qa_set if k)
    print(f"{ratio:>8.1f} {tok:>10} {hits/n:>7.0%}")
```

위 실행에서는 세 압축률 모두 정답률 100%였고, 0.4의 토큰 수(254)가 0.6(261)과 거의 같습니다. 모델이 목표 글자 수를 지키지 않은 것입니다. 정답률이 안 떨어진 데는 판정이 느슨한 탓도 있습니다. 질문마다 핵심어 하나("이월", "10")만 찾으므로 "인사팀장 승인", "최대 5일" 같은 조건절이 요약에서 빠져도 정답으로 셉니다.

요약문을 직접 출력해 조건·예외·숫자가 남았는지 읽고, 판정을 조건어까지 요구하도록(이월 질문은 "인사팀장"과 "5일" 둘 다) 바꿔 다시 재 보십시오. 압축률을 낮추면 **조건절과 숫자가 먼저 사라지는** 것이 그때 숫자로 보입니다. 요약 프롬프트에 "조건, 예외, 숫자는 유지하라"를 넣어도 완전히 막지는 못합니다.

압축은 공짜가 아닙니다. 규정 문서처럼 세부 조건이 핵심인 문서에서는 위험하고 배경 설명이 긴 문서에서는 효과적입니다. **문서 성격에 따라 판단하십시오.**

**실행 순서** : 정답 코드를 실습 셀 끝에 붙여 다시 실행하십시오. `compress`가 정의된 뒤에 실행되어야 합니다. 조항마다 요약 호출이 한 번씩 들어가므로 압축률 두 가지에 2~4분이 걸립니다.

</details>

#### 트랙 4 : 서브에이전트 분할

큰 작업을 쪼개 각각 별도 컨텍스트를 가진 에이전트에 맡기면 각자의 컨텍스트 윈도우가 깨끗해집니다.

In [ ]:
# 실습 : 복합 질문을 하위 질문으로 쪼개고, 각각 독립 컨텍스트에서 답한 뒤 종합하는 구조를 만드세요
# 요구사항
#  1) 분해 : 복합 질문 → 하위 질문 목록
#  2) 각 하위 질문을 별도 호출로 처리 (히스토리 공유 없음)
#  3) 종합 : 하위 답변들을 모아 최종 답변 작성
#  4) 단일 컨텍스트로 한 번에 답한 결과와 비교

complex_q = "신입사원이 입사 첫 달에 알아야 할 근무 시간, 재택근무 신청 방법, 연차 관련 규정을 정리해주세요."

def subagent_answer(question):
    # 여기에 분해, 개별 처리, 종합을 작성하세요
    return ""          # 아직 구현 전이라도 뒤 셀이 오류로 멈추지 않게 둡니다

<details>
<summary>정답 보기</summary>

```python
def subagent_answer(question):
    # 1) 분해
    plan = chat([{"role": "system", "content":
                  "복합 질문을 독립적으로 답할 수 있는 하위 질문들로 쪼개세요. "
                  "하위 질문은 반드시 한국어로 작성하세요. "
                  'JSON 배열로만 출력하세요. 형식: ["질문1", "질문2"]'},
                 {"role": "user", "content": question}], max_new_tokens=300, temperature=0.0)
    m = re.search(r"\[.*\]", plan, re.S)
    # 분해가 깨지면 예외를 내지 않고 원 질문 하나로 되돌립니다.
    # 여기서 예외를 던지면 뒤 단계가 통째로 날아갑니다
    try:
        subs = json.loads(m.group(0)) if m else [question]
    except json.JSONDecodeError:
        subs = [question]
    if not isinstance(subs, list) or not subs:
        subs = [question]
    print(f"하위 질문 {len(subs)}개 : {subs}")

    # 2) 각각 독립 컨텍스트에서 처리
    partials = []
    for s in subs:
        ctx = "\n\n".join(chunks)
        a = chat([{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {s}"}],
                 max_new_tokens=200, temperature=0.0)
        partials.append(f"[{s}]\n{a}")

    # 3) 종합. 규정 원문은 넣지 않습니다. 컨텍스트 윈도우가 깨끗해집니다
    final = chat([{"role": "system", "content":
                   "하위 답변들을 종합해 하나의 정리된 안내문을 작성하세요. "
                   "근거 조항 번호를 유지하세요. 답변은 반드시 한국어로만 작성하세요."},
                  {"role": "user", "content": "\n\n".join(partials)
                                              + f"\n\n원래 질문: {question}"}],
                 max_new_tokens=500)
    return final
```

종합 단계에는 **규정 원문을 다시 넣지 않습니다.** 하위 답변에 필요한 정보가 이미 추출되어 있으므로 종합 에이전트의 컨텍스트 윈도우는 훨씬 깨끗합니다.

분해가 깨지면 예외를 던지지 않고 원 질문 하나짜리 목록으로 되돌립니다. 분해는 **있으면 좋은 단계**일 뿐이라 여기서 예외를 내면 답변 자체가 안 나갑니다.

**실행 순서** : 정답 코드를 실습 셀에 붙여 다시 실행한 다음 요구사항 4번(단일 컨텍스트와의 비교) 셀을 실행하십시오.

</details>

In [ ]:
# 요구사항 4 : 단일 컨텍스트로 한 번에 답한 결과와 비교합니다
print("=== 쪼개서 답하기 ===")
multi = subagent_answer(complex_q)
print(multi[:400])

print("\n=== 한 번에 답하기 ===")
single = chat([{"role": "system", "content": SYSTEM},
               {"role": "user", "content": "[규정]\n" + "\n\n".join(chunks)
                                           + f"\n\n질문: {complex_q}"}],
              max_new_tokens=500, temperature=0.0)
print(single[:400])

# 세 주제의 근거 조항이 각각 살아남았는지 셉니다.
# "다뤘는가"가 아니라 "근거가 붙었는가"로 재야 종합 단계의 손실이 보입니다
GROUND = ["제3조", "제5조", "제7조"]      # 근무 시간, 재택근무, 연차 이월

print(f"\n{'방식':<12} {'길이':>6} {'근거 조항':>10}")
print("-" * 32)
for name, txt in [("쪼개기", multi), ("한 번에", single)]:
    kept = sum(int(k in txt) for k in GROUND)
    print(f"{name:<12} {len(txt):>6} {kept:>9}/{len(GROUND)}")

=== 쪼개서 답하기 ===
하위 질문 3개 : ['신입사원이 입사 첫 달에 알아야 할 근무 시간은 어떻게 되나요?', '재택근무를 신청하는 방법은 무엇인가요?', '연차 관련 규정은 어떤 것이 있나요?']
신입사원이 입사 첫 달에 알아야 할 근무 시간, 재택근무 신청 방법, 그리고 연차 관련 규정을 정리하면 다음과 같습니다:

1. **근무 시간**  
   - 주 40시간으로, 1일 근무 시간은 휴게 시간을 제외하고 8시간으로 정해져 있습니다.
   - 일반적인 경우, 오전 10시부터 오후 7시까지 근무합니다. (참조 조항: 제3조)

2. **재택근무 신청 방법**  
   - 주 2회 이내에서 재택근무를 신청해야 합니다.
   - 근무일 전일 오후 6시까지 부서장의 승인을 받아야 합니다. (규정 제5조 참조)

3. **연차 관련 규정**  
   - 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음이 원칙입니다.
   - 회사의 업무상 사유로 사용하지 못한 경우, 인사팀장의 승인을 받

=== 한 번에 답하기 ===
신입사원이 입사 첫 달에 알아야 할 근무 시간, 재택근무 신청 방법, 연차 관련 규정은 다음과 같습니다.

### 근무 시간
- **기본 근무 시간**: 주 40시간으로, 1일 근무 시간은 휴게 시간을 제외하고 8시간입니다. 시업 시각은 오전 10시, 종업 시각은 오후 7시입니다. 유연근무제 적용자는 제4조에 따라 오전 8시부터 오전 11시 사이에 시업 시각을 선택할 수 있으며, 코어 타임은 오전 11시부터 오후 4시까지입니다.

### 재택근무 신청 방법
- **재택근무 가능 횟수**: 임직원은 주 2회 이내에서 재택근무를 신청할 수 있습니다.
- **신청 절차**: 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 합니다.

### 연차 관련 규정
- **연차 휴가의 이월**: 당해 연도

방식               길이      근거 조항
--------------------------------

#### 트랙 5 : 프롬프트 캐시 관점의 순서 설계

비용 최적화 트랙입니다. 대부분의 추론 엔진은 **프롬프트 앞부분이 같으면 계산을 재사용**하며 이것을 프롬프트 캐시 또는 프리픽스 캐시라고 합니다. 그래서 변하지 않는 것을 앞에, 매번 변하는 것을 뒤에 두면 캐시 적중률이 올라갑니다.

In [ ]:
# 실습 : 두 가지 배치 순서로 같은 질문 세트를 처리하고 총 시간을 비교하세요
#  배치 A : [시스템][규정 원문][질문]      - 고정된 것이 앞
#  배치 B : [질문][시스템][규정 원문]      - 매번 변하는 것이 앞
# 로컬 transformers는 캐시가 제한적이지만, 경향을 관찰합니다

def timed_run(order):
    ctx = "\n\n".join(chunks)
    start = time.time()
    for q, _ in qa_set:
        if order == "A":
            msgs = [{"role": "system", "content": SYSTEM},
                    {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
        else:
            msgs = [{"role": "system", "content": f"질문: {q}\n\n" + SYSTEM},
                    {"role": "user", "content": f"[규정]\n{ctx}"}]
        chat(msgs, max_new_tokens=150, temperature=0.0)
    return time.time() - start


# 여기에 두 배치를 비교하는 코드를 작성하세요

배치 A : 16.1초 (3.23초/질문)
배치 B : 21.9초 (4.39초/질문)

<details>
<summary>정답 보기</summary>

```python
for order in ["A", "B"]:
    t = timed_run(order)
    print(f"배치 {order} : {t:.1f}초 ({t/len(qa_set):.2f}초/질문)")
```

로컬 `transformers`는 호출 사이에 프리픽스 캐시를 재사용하지 않으므로 두 배치의 시간 차이는 캐시 효과가 아닙니다. 위 실행의 16.1초와 21.9초 차이는 질문마다 다른 답변 길이(최대 150토큰 안에서)와 측정 잡음에서 온 것이라 배치 A가 빨랐다고 캐시가 동작했다고 읽으면 안 됩니다. 여기서는 **원리를 아는 것**이 목적입니다. vLLM 같은 서빙 엔진에서는 프리픽스 캐시가 기본입니다. 배치 A처럼 고정 부분을 앞에 두면 재계산을 건너뛰어 지연이 크게 줄어듭니다. 사내 문서 검색 서비스에서 시스템 프롬프트와 자주 쓰이는 문서를 앞에 고정하는 설계가 여기서 나옵니다.

이 주제는 서빙 심화 주간에서 **실제 캐시 적중률을 측정하며** 다시 다룹니다. 이론 레슨 3.7절에서 같은 약속을 해두었습니다.

**실행 순서** : 정답 코드를 실습 셀 끝에 붙여 다시 실행하십시오. `timed_run`이 정의된 뒤에 실행되어야 합니다. 배치 두 가지에 5문항씩 총 10회 생성이라 2~4분이 걸립니다.

</details>

## Step 7 : 공유와 정리

각자 수행한 두 트랙을 공유합니다.

**1. 선택한 트랙과 이유**

**2. 측정 결과** : 무엇을 바꿨고 숫자가 어떻게 변했는지

**3. 어느 층위의 문제였는지** : 여러분이 다룬 것이 2층인지 3층인지 4층인지, 그리고 왜 그 층에서 해결해야 했는지

**4. 프롬프트만으로는 해결할 수 없었던 이유**

**5. 정지 조건 세어보기** : 트랙에서 루프를 만들었다면 성공·한계·정체·위임 중 몇 개가 실제 코드에 있었는지

Step 4에서 **모델은 하나도 바뀌지 않았는데 규약 준수율이 달라졌습니다.** 가이드를 주고 센서를 붙이고 실패를 피드백으로 되돌려 준 것뿐입니다.

우리는 지난 여섯 노드를 네 층으로 다시 읽었습니다. RAG는 컨텍스트 엔지니어링이었고 에이전트는 하네스였고 재시도와 루프 감지는 루프 엔지니어링이었습니다.

앞으로 남은 커리큘럼도 이 틀로 읽을 수 있습니다. 파인튜닝은 모델 자체를 바꾸는 유일한 층입니다. 서빙과 최적화는 하네스를 빠르게 돌리는 기술이고 모니터링과 평가는 센서를 운영 환경으로 확장하는 일입니다.

다음 노드 멀티모달 RAG에서는 컨텍스트에 **텍스트가 아닌 것**을 넣습니다. 이미지가 섞인 문서를 다루며 2층의 범위를 넓힙니다.

## 마무리 과제 : 나의 태스크에 하네스 붙이기

여러분이 정한 태스크에 하네스를 설계하고 하네스만으로 품질이 얼마나 오르는지 측정하십시오.

**1단계. 태스크와 규약 정의**

1. 반복적으로 수행할 태스크를 하나 고릅니다. (커밋 메시지 작성, 회의록 정리, 코드 리뷰 코멘트, 상품 설명문 등)  
2. 그 태스크의 **가이드**를 작성합니다. 규약 5개 이상, 기계적으로 검사 가능한 항목을 포함해야 합니다.  
3. 테스트 입력 8개 이상을 준비합니다.

**2단계. 센서 구현**

4. 규칙 기반 센서를 구현합니다. 규약 위반을 구체적 메시지로 반환해야 합니다.  
5. 판정 모델 센서를 추가합니다. 규칙으로 잡을 수 없는 항목을 담당합니다.

**3단계. 측정**

6. 세 조건의 통과율을 측정합니다.

| 조건 | 통과율 | 평균 시도 횟수 |
|---|---|---|
| 가이드 없음, 센서 없음 | | 1 |
| 가이드만 | | 1 |
| 가이드 + 센서 + 피드백 루프 | | |

7. 검증 루프의 수렴 곡선을 측정합니다. 최대 반복 1, 2, 3, 5회별 통과율.

**4단계. 리포트**

- **정체 지점** : 몇 회부터 개선이 멈췄고 그 이유는 무엇이라고 보는지  
- **끝까지 통과하지 못한 케이스** : 루프를 늘려서 해결될 문제인지, 가이드를 고쳐야 할 문제인지 판단  
- **규칙 센서가 놓친 것과 판정 센서가 놓친 것** 각각 하나 이상

**평가 기준**

최종 통과율이 아니라 **층위 판단**으로 평가합니다. 문제를 만났을 때 프롬프트를 고칠 문제인지, 컨텍스트 구성을 바꿀 문제인지, 센서를 추가할 문제인지, 루프 구조를 바꿀 문제인지 구분해 설명할 수 있으면 좋은 리포트입니다.